In [1]:
import os 
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HOME"] = "/scratch/adelmou/hf_home/"
os.environ["HF_HUB_CACHE"] = "/scratch/adelmou/hf_home/hub/"

import torch 

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True


In [2]:
import speechbrain as sb
from speechbrain.utils.logger import get_logger
from speechbrain.dataio.dataloader import LoopedLoader
from speechbrain.dataio.dataloader import DataLoader
import torch
import numpy as np
from tqdm import tqdm
from abc import ABCMeta, abstractmethod
from pathlib import Path
from contextlib import ExitStack
from typing import Dict

logger = get_logger(__name__)


In [31]:

class FeaturesReader(metaclass=ABCMeta):
    @property
    @abstractmethod
    def name(self) -> str:
        ...

    @abstractmethod
    def read(
        self,
        key: str,
        left_offset_frames: int = 0,
        right_offset_frames: int = None,
    ) -> np.ndarray:
        ...

class FeaturesWriter(metaclass=ABCMeta):
    @property
    @abstractmethod
    def name(self) -> str:
        ...

    @property
    @abstractmethod
    def storage_path(self) -> str:
        ...

    @abstractmethod
    def write(self, key: str, value: np.ndarray) -> str:
        ...
    def __enter__(self):
        return self

    def __exit__(self, *args, **kwargs):
        ...


class NumpyHdf5Writer(FeaturesWriter):
    name = "numpy_hdf5"

    def __init__(self, storage_path: Path, mode: str = "w", dtype=np.float32, *args, **kwargs):
        """
        :param storage_path: Path under which we'll create the HDF5 file.
            We will add a ``.h5`` suffix if it is not already in ``storage_path``.
        :param mode: Modes supported by h5py:
            w        Create file, truncate if exists (default)
            w- or x  Create file, fail if exists
            a        Read/write if exists, create otherwise
        """
        super().__init__()
        import h5py

        p = Path(storage_path)
        self.storage_path_ = p.with_suffix(
            p.suffix + ".h5" if p.suffix != ".h5" else ".h5"
        )
        print(f"saving to {self.storage_path_}")
        self.hdf = h5py.File(self.storage_path, mode=mode)
        self.dtype = dtype

    @property
    def storage_path(self) -> str:
        return str(self.storage_path_)

    def write(self, key: str, value: np.ndarray) -> str:
        value = value.cpu().numpy().astype(self.dtype)
        self.hdf.create_dataset(key, data=value)
        return key

    def close(self) -> None:
        return self.hdf.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()

class NumpyHdf5Reader(FeaturesReader):

    name = "numpy_hdf5"

    def __init__(self, storage_path: Path, *args, **kwargs):
        super().__init__()
        import h5py
        self.hdf = h5py.File(storage_path, mode="r")

    def read(
        self,
        key: str,
        left_offset_frames: int = 0,
        right_offset_frames: int = None,
    ) -> np.ndarray:
        return self.hdf[key][left_offset_frames:right_offset_frames]

    def close(self) -> None:
        return self.hdf.close()

import lilcom
class LilcomHdf5Reader(FeaturesReader):
    """
    Reads lilcom-compressed numpy arrays from a HDF5 file with a "flat" layout.
    Each array is stored as a separate HDF ``Dataset`` because their shapes (numbers of frames) may vary.
    ``storage_path`` corresponds to the HDF5 file path;
    ``storage_key`` for each utterance is the key corresponding to the array (i.e. HDF5 "Group" name).
    """

    name = "lilcom_hdf5"

    def __init__(self, storage_path: Path, *args, **kwargs):
        super().__init__()
        import h5py
        self.hdf = h5py.File(storage_path, mode="r")

    def read(
        self,
        key: str,
        left_offset_frames: int = 0,
        right_offset_frames: int = None,
    ) -> np.ndarray:
        arr = lilcom.decompress(self.hdf[key][()].tobytes())
        return arr[left_offset_frames:right_offset_frames]
    
class LilcomHdf5Writer(FeaturesWriter):
    """
    Writes lilcom-compressed numpy arrays to a HDF5 file with a "flat" layout.
    Each array is stored as a separate HDF ``Dataset`` because their shapes (numbers of frames) may vary.
    ``storage_path`` corresponds to the HDF5 file path;
    ``storage_key`` for each utterance is the key corresponding to the array (i.e. HDF5 "Group" name).
    """

    name = "lilcom_hdf5"

    def __init__(
        self,
        storage_path: Path,
        dtype=np.float32,
        tick_power: int = -5,
        mode: str = "w",
        *args,
        **kwargs,
    ):
        """
        :param storage_path: Path under which we'll create the HDF5 file.
            We will add a ``.h5`` suffix if it is not already in ``storage_path``.
        :param tick_power: Determines the lilcom compression accuracy;
            the input will be compressed to integer multiples of 2^tick_power.
        :param mode: Modes supported by h5py:
            w        Create file, truncate if exists (default)
            w- or x  Create file, fail if exists
            a        Read/write if exists, create otherwise
        """
        super().__init__()
        import h5py

        p = Path(storage_path)
        self.storage_path_ = p.with_suffix(
            p.suffix + ".h5" if p.suffix != ".h5" else ".h5"
        )
        self.hdf = h5py.File(self.storage_path, mode=mode)
        self.tick_power = tick_power
        self.dtype = dtype

    @property
    def storage_path(self) -> str:
        return str(self.storage_path_)

    def write(self, key: str, value: np.ndarray) -> str:
        value = value.cpu().numpy().astype(self.dtype)
        serialized_feats = lilcom.compress(value, tick_power=self.tick_power)
        self.hdf.create_dataset(key, data=np.void(serialized_feats))
        return key

    def close(self) -> None:
        return self.hdf.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()

In [19]:
sample_rate = 16000
n_fft = 512
n_mels = 80
win_length = 32
fbank = sb.lobes.features.Fbank(
   sample_rate=sample_rate,
   n_fft=n_fft,
   n_mels=n_mels,
   win_length=win_length
)

In [32]:
class ExtractFeatures(sb.core.Brain):
    def compute_features(self, batch):
        #NOTE: the user will only have to implement this method
        batch = batch.to(self.device)
        wavs, wav_lens = batch.sig
        feats = fbank(wavs)
        batch_size = wavs.shape[0]
        codecs = torch.randint(0, 2048, (batch_size, 32, 512))
        
        return [{"id": batch.id[i], "feats": feats[i], "codecs": codecs[i]} for i in range(batch_size)]
    
    def cache_features(
        self,
        writers: Dict[str, FeaturesWriter],
        dataset,
        loader_kwargs={},
        progressbar=None,
    ):
        #NOTE: will be part of core.py
        assert writers is not None, "Feature cache manager must be provided"
        
        if not (
            isinstance(dataset, DataLoader)
            or isinstance(dataset, LoopedLoader)
        ):
            dataset = self.make_dataloader(
                dataset, stage=sb.Stage.TEST, **loader_kwargs
            )
        
        if progressbar is None:
            progressbar = not self.noprogressbar

        # Only show progressbar if requested and main_process
        enable = progressbar and sb.utils.distributed.if_main_process()
        feature_types = list(writers.keys())
        with ExitStack() as stack:
            open_writers = {ft: stack.enter_context(writers[ft]) for ft in writers.keys()}
            for data in self._cache_features(
                dataset=dataset,
                enable=enable,
            ):
                for item in data:
                    for ft in feature_types:
                        open_writers[ft].write(item['id'], item[ft])


    def _cache_features(self, dataset, enable):
        #NOTE: will be part of core.py
        for batch in tqdm(dataset, dynamic_ncols=True, disable=not enable, colour=self.tqdm_barcolor["valid"]):
            extracted_features = self.compute_features(batch)
            yield extracted_features
    

In [5]:
import os 
from librispeech_prepare import prepare_librispeech  # noqa
prepare_librispeech(
    data_folder= os.environ["SLURM_TMPDIR"] + "/LibriSpeech",
    tr_splits= ['train-clean-100'],
    dev_splits= ['dev-clean'],
    te_splits= ['test-clean'],
    save_folder= os.environ["SLURM_TMPDIR"] + "/save",
    merge_lst= ['train-clean-100'],
    merge_name= "train.csv",
    skip_prep= False,
)

  0%|          | 0/28539 [00:00<?, ?it/s]

  0%|          | 0/2703 [00:00<?, ?it/s]

  0%|          | 0/2620 [00:00<?, ?it/s]

In [23]:
def dataio_prepare():
    """This function prepares the datasets to be used in the brain class.
    It also defines the data processing pipeline through user-defined functions.
    """
    data_folder = os.environ["SLURM_TMPDIR"] + "/save"

    train_data = sb.dataio.dataset.DynamicItemDataset.from_csv(
        csv_path=os.environ["SLURM_TMPDIR"] + "/save/train.csv",
        replacements={"data_root": data_folder},
    )

    datasets = [train_data]

    # 2. Define audio pipeline:
    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline(wav):
        sig = sb.dataio.dataio.read_audio(wav)
        return sig

    sb.dataio.dataset.add_dynamic_item(datasets, audio_pipeline)

    # 4. Set output:
    sb.dataio.dataset.set_output_keys(
        datasets,
        ["id", "sig"],
    )

    return (
        train_data,
    )

train_data, = dataio_prepare()

feature_types = ['feats', 'codecs']
feature_dtypes = [np.float32, np.int16]
writers = {
    ft: LilcomHdf5Writer(
        storage_path=os.path.join(os.environ['SLURM_TMPDIR'], f"save/train_lilcom_{ft}.h5"),
        dtype=feature_dtypes[idx]
    )
    for idx, ft in enumerate(feature_types)
}

extractor = ExtractFeatures()
dataloader = sb.dataio.dataloader.make_dataloader(
    train_data, **{"batch_size": 12}
)

In [33]:
feature_types = ['feats', 'codecs']
feature_dtypes = [np.float32, np.int16]
writers = {
    ft: LilcomHdf5Writer(
        storage_path=os.path.join(os.environ['SLURM_TMPDIR'], f"save/train_lilcom_{ft}.h5"),
        dtype=feature_dtypes[idx]
    )
    for idx, ft in enumerate(feature_types)
}

extractor = ExtractFeatures()
dataloader = sb.dataio.dataloader.make_dataloader(
    train_data, **{"batch_size": 12}
)

The model has no parameters!


In [34]:
extractor.cache_features(writers, dataloader)

100%|███████████████████████████████████████████████████████████████████████| 2379/2379 [06:11<00:00,  6.41it/s]


In [35]:
reader_manager = LilcomHdf5Reader(storage_path=os.environ["SLURM_TMPDIR"] + "/save/train_lilcom_feats.h5")
for idx, i in enumerate(train_data):
    print(i['id'])
    print(reader_manager.read(i['id']))
    if idx > 10:
        break
reader_manager.close()

83-11691-0004
[[-20.53125  -20.809082 -21.362068 ... -50.345634 -48.37767  -49.210693]
 [-17.24585  -17.319633 -17.415565 ... -50.320663 -50.111397 -49.6476  ]
 [-20.298616 -20.000626 -19.279907 ... -46.873985 -45.2173   -47.66456 ]
 ...
 [-50.33953  -50.3425   -50.34616  ... -50.317398 -50.31903  -50.328056]
 [-50.321255 -50.3425   -50.32849  ... -50.318504 -50.31746  -50.325745]
 [-50.33437  -50.319656 -50.327576 ... -50.318726 -50.34729  -50.34256 ]]
83-11691-0035
[[-25.90625  -25.90271  -25.77551  ... -41.756695 -41.309822 -49.626022]
 [-20.782715 -20.780994 -20.654716 ... -42.68499  -42.363235 -48.412155]
 [-20.207985 -20.007343 -19.477173 ... -45.06926  -45.443542 -48.005417]
 ...
 [-13.987492 -15.292683 -21.102325 ... -51.039032 -51.021187 -51.017   ]
 [ -9.081437 -10.631659 -21.625858 ... -51.031593 -51.044388 -51.016953]
 [-12.31454  -13.395273 -17.170174 ... -51.01497  -51.02565  -51.03653 ]]
83-11691-0031
[[-25.3125   -25.746094 -26.79419  ... -51.6408   -51.6593   -51.64077

AttributeError: 'LilcomHdf5Reader' object has no attribute 'close'

In [12]:
os.environ["SLURM_TMPDIR"] + "/save/train_codecs.h5"

'/localscratch/adelmou.45643353.0/save/train_codecs.h5'

In [36]:
!ls -lh '/localscratch/adelmou.45643353.0/save/'

total 17G
-rw-r-----. 1 adelmou adelmou 608K Jul  1 14:42 dev-clean.csv
-rw-r-----. 1 adelmou adelmou   37 Jul  1 14:42 opt_librispeech_prepare.pkl
-rw-r-----. 1 adelmou adelmou 590K Jul  1 14:42 test-clean.csv
-rw-r-----. 1 adelmou adelmou 8.6M Jul  1 14:42 train-clean-100.csv
-rw-r-----. 1 adelmou adelmou 903M Jul  1 15:06 train_codecs.h5
-rw-r-----. 1 adelmou adelmou 8.6M Jul  1 14:42 train.csv
-rw-r-----. 1 adelmou adelmou  14G Jul  1 15:06 train_feats.h5
-rw-r-----. 1 adelmou adelmou 965M Jul  1 15:22 train_lilcom_codecs.h5
-rw-r-----. 1 adelmou adelmou 4.8G Jul  1 15:22 train_lilcom_feats.h5
